In [10]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [11]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

from pprint import pformat

formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))

<function __main__.<lambda>(obj, p, _)>

<function __main__.<lambda>(obj, p, _)>

In [12]:
from datasets import load_from_disk

mquake = load_from_disk("../data/mquake")
mquake
mquake[0]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'portability'],
    num_rows: 2446
})

{'subject': 'Carl Sagan',
 'prompt': 'Who is the employer of Carl Sagan?',
 'old_ans': 'Cornell University',
 'new_ans': 'British Broadcasting Corporation',
 'portability': [{'hop': {'new': [{'ans': 'Carl Sagan', 'prompt': 'Who is Ann Druyan married to?'},
                                  {'ans': 'British Broadcasting Corporation',
                                   'prompt': 'Who is the employer of Carl Sagan?'}],
                          'old': [{'ans': 'Carl Sagan', 'prompt': 'Who is Ann Druyan married to?'},
                                  {'ans': 'Cornell University',
                                   'prompt': 'Who is the employer of Carl Sagan?'}]},
                  'new_ans': 'British Broadcasting Corporation',
                  'old_ans': 'Cornell University',
                  'prompt': 'Who is the employer of the spouse of Ann Druyan?',
                  'rephrases': ['Where does the spouse of Ann Druyan work?',
                                'What company employs the

In [ ]:
import random
from dataclasses import asdict
from src.utils import EditRequest, EvalPair, EvalSuite
from evaluate.evaluate_temp import save_results

n = 20  # number of samples — change as needed

mquake_indices = random.sample(range(len(mquake)), n)
print(f"mquake indices: {mquake_indices}")

mquake indices: [1229, 2397, 1556, 254, 102, 976, 29, 767, 234, 450, 313, 1328, 1915, 1376, 2208, 304, 2065, 623, 494, 1043]


In [14]:
from src.editors import load_editor
editor = load_editor("../src/config/grace/gpt2-xl.yaml")

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

In [15]:
mquake_results = []

for i, idx in enumerate(mquake_indices):
    sample = mquake[idx]
    request = EditRequest(
        subject=sample["subject"],
        prompt=sample["prompt"],
        old_ans=sample["old_ans"],
        new_ans=sample["new_ans"],
    )
    editor.edit(request, is_sequential=True)

    # Portability: prompt + rephrases from each portability entry
    port_pairs = []
    for port in sample["portability"]:
        port_pairs.append(EvalPair(prompt=port["prompt"], ans=port["new_ans"]))
        for rephrase in port["rephrases"]:
            port_pairs.append(EvalPair(prompt=rephrase, ans=port["new_ans"]))

    suite = EvalSuite(
        portability={"portability": port_pairs} if port_pairs else None,
    )
    result = editor.eval(suite, max_new_tokens=50)

    mquake_results.append({
        "edit_index": i + 1,
        "dataset_index": idx,
        "edit_request": asdict(request),
        **result.to_dict(),
    })
    port_score = (list(result.portability.values())[0].score
                  if result.portability else "N/A")
    print(f"[mquake {i+1}/{n}] portability={port_score}")

print("mquake phase done.")

[mquake 1/20] portability=0.0
[mquake 2/20] portability=0.0
[mquake 3/20] portability=0.0
[mquake 4/20] portability=0.0
[mquake 5/20] portability=0.0
[mquake 6/20] portability=0.0
[mquake 7/20] portability=0.0
[mquake 8/20] portability=0.0
[mquake 9/20] portability=0.0
[mquake 10/20] portability=0.0
[mquake 11/20] portability=0.0
[mquake 12/20] portability=0.0
[mquake 13/20] portability=0.0
[mquake 14/20] portability=0.0
[mquake 15/20] portability=0.0
[mquake 16/20] portability=0.0
[mquake 17/20] portability=0.0
[mquake 18/20] portability=0.0
[mquake 19/20] portability=0.3333333333333333
[mquake 20/20] portability=0.0
mquake phase done.


In [16]:
mquake_results

[{'edit_index': 1,
  'dataset_index': 1229,
  'edit_request': {'subject': 'starting pitcher',
                   'prompt': 'Which sport is starting pitcher associated with?',
                   'old_ans': 'baseball',
                   'new_ans': 'rugby union'},
  'portability': {'portability': {'score': 0.0,
                                  'details': [{'prompt': 'What sport is associated with the '
                                                         'position played by Randy Wolf?',
                                               'expected': 'rugby union',
                                               'generated': '\n'
                                                            '\n'
                                                            'Randy Wolf was a professional '
                                                            'wrestler who was born in the United '
                                                            'States. He was a member of the World '
        

In [17]:
filepath = save_results(
    {"mquake": mquake_results},
    editor.config.model_name,
    editor.config.alg_name,
    output_dir="../results/mquake",
)
print(f"Results saved to: {filepath}")

Results saved to: ../results/mquake\gpt2-xl_GRACE_2026-03-05_15-31.json


In [18]:
editor.close()
del editor